In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer 

In [7]:
# Hospital patient data: Age, Weight(kg), BloodPressure, Cholesterol
data = {
    'Age': [25, 30, np.nan, 45, 50, np.nan, 35, 60, 28, 55],
    'Weight': [70, 80, 65, np.nan, 90, 75, 68, 85, 72, np.nan],
    'BloodPressure': [120, 130, 115, 140, np.nan, 125, 118, 145, 122, 138],
    'Cholesterol': [200, 220, np.nan, 250, 240, 210, np.nan, 260, 205, 245]
}

df = pd.DataFrame(data)

In [13]:
df.head()

,Age,Weight,BloodPressure,Cholesterol
0,25.0,70.0,120.0,200.0
1,30.0,80.0,130.0,220.0
2,NaN,65.0,115.0,NaN
3,45.0,NaN,140.0,250.0
4,50.0,90.0,NaN,240.0


In [18]:
df.isnull().sum().sum()

np.int64(7)

KNN APPLY

In [20]:
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

x_imputed = knn_imputer.fit_transform(df) 

df_imputed = pd.DataFrame(x_imputed, columns=df.columns) 

print("=== AFTER KNN IMPUTATION ===")
print(df_imputed.round(2))

=== AFTER KNN IMPUTATION ===
     Age  Weight  BloodPressure  Cholesterol
0  25.00   70.00         120.00       200.00
1  30.00   80.00         130.00       220.00
2  31.87   65.00         115.00       209.05
3  45.00   81.16         140.00       250.00
4  50.00   90.00         136.55       240.00
5  30.44   75.00         125.00       210.00
6  35.00   68.00         118.00       211.05
7  60.00   85.00         145.00       260.00
8  28.00   72.00         122.00       205.00
9  55.00   82.83         138.00       245.00


In [21]:
print("\n=== COMPARISON: Before vs After ===")
for col in df.columns:
    missing_mask = df[col].isnull()
    if missing_mask.any():
        print(f"\n📌 Column: {col}")
        missing_rows = missing_mask[missing_mask].index.tolist()
        for row in missing_rows:
            print(f"   Row {row}: was NaN → now {df_imputed.loc[row, col]:.2f}")



=== COMPARISON: Before vs After ===

📌 Column: Age
   Row 2: was NaN → now 31.87
   Row 5: was NaN → now 30.44

📌 Column: Weight
   Row 3: was NaN → now 81.16
   Row 9: was NaN → now 82.83

📌 Column: BloodPressure
   Row 4: was NaN → now 136.55

📌 Column: Cholesterol
   Row 2: was NaN → now 209.05
   Row 6: was NaN → now 211.05


In [22]:
from sklearn.impute import SimpleImputer

# Simple approach: just use column mean
simple_imputer = SimpleImputer(strategy='mean')
X_simple = simple_imputer.fit_transform(df)
df_simple = pd.DataFrame(X_simple, columns=df.columns)

print("=== COMPARISON TABLE ===")
for col in df.columns:
    missing_mask = df[col].isnull()
    if missing_mask.any():
        print(f"\n📌 {col}:")
        print(f"   Mean imputation:  {df_simple.loc[missing_mask, col].values.round(2)}")
        print(f"   KNN imputation:   {df_imputed.loc[missing_mask, col].values.round(2)}")
        print(f"   ↑ KNN is smarter because it considers SIMILAR patients, not ALL patients")


=== COMPARISON TABLE ===

📌 Age:
   Mean imputation:  [41. 41.]
   KNN imputation:   [31.87 30.44]
   ↑ KNN is smarter because it considers SIMILAR patients, not ALL patients

📌 Weight:
   Mean imputation:  [75.62 75.62]
   KNN imputation:   [81.16 82.83]
   ↑ KNN is smarter because it considers SIMILAR patients, not ALL patients

📌 BloodPressure:
   Mean imputation:  [128.11]
   KNN imputation:   [136.55]
   ↑ KNN is smarter because it considers SIMILAR patients, not ALL patients

📌 Cholesterol:
   Mean imputation:  [228.75 228.75]
   KNN imputation:   [209.05 211.05]
   ↑ KNN is smarter because it considers SIMILAR patients, not ALL patients


In [24]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

# Small enough to verify by hand
data = {
    'Age':    [25, 30, np.nan, 45],
    'Weight': [70, 80, 65,     90],
    'BP':     [120, 130, 115,  140]
}
df = pd.DataFrame(data)
print("BEFORE:")
print(df)

# n_neighbors=2: look at 2 closest patients (small data, can't do 5)
# weights='distance': closer patients count more
knn_imputer = KNNImputer(n_neighbors=2, weights='distance')

X_imputed = knn_imputer.fit_transform(df)
df_filled = pd.DataFrame(X_imputed, columns=df.columns)

print("\nAFTER:")
print(df_filled)

# NOW VERIFY BY HAND:
# Patient 2 missing Age. Has Weight=65, BP=115
# Dist to Patient 0: sqrt((65-70)² + (115-120)²) = sqrt(50) = 7.07
# Dist to Patient 1: sqrt((65-80)² + (115-130)²) = sqrt(450) = 21.21
# Dist to Patient 3: sqrt((65-90)² + (115-140)²) = sqrt(1250) = 35.36
# Top 2 nearest: Patient 0 (d=7.07), Patient 1 (d=21.21)
# w0 = 1/7.07 = 0.1414,  w1 = 1/21.21 = 0.0471
# Estimated Age = (0.1414*25 + 0.0471*30) / (0.1414+0.0471) = 26.33
print("\nManual calculation: should be ≈ 26.33")


BEFORE:
    Age  Weight   BP
0  25.0      70  120
1  30.0      80  130
2   NaN      65  115
3  45.0      90  140

AFTER:
     Age  Weight     BP
0  25.00    70.0  120.0
1  30.00    80.0  130.0
2  26.25    65.0  115.0
3  45.00    90.0  140.0

Manual calculation: should be ≈ 26.33
